# 21 — Traffic Insights Dashboard Queries

## Query 1: Monthly pollution/noise trend, top 10 worst streets

In [0]:
top10_trend = spark.sql("""
SELECT
    t.month_year,
    t.street_name,
    t.avg_pollution,
    t.avg_noise,
    t.reading_count
FROM vstone_catalog.gold.agg_monthly_street_trend t
JOIN (
    SELECT street_id FROM vstone_catalog.gold.agg_top10_streets_by_pollution
) top ON t.street_id = top.street_id
ORDER BY t.month_year, t.avg_pollution DESC
""")
display(top10_trend)

## Query 2: Top 10 streets leaderboard (single snapshot)

In [0]:
top10_snapshot = spark.sql("""
SELECT street_name, avg_pollution, avg_noise, reading_count
FROM vstone_catalog.gold.agg_top10_streets_by_pollution
ORDER BY avg_pollution DESC
""")
display(top10_snapshot)

## Query 3: Daily traffic volume by sensor node (map-ready — has lat/long)

In [0]:
traffic_by_node = spark.sql("""
SELECT
    location, latitude, longitude,
    SUM(total_enter) AS total_enter,
    SUM(total_exit) AS total_exit,
    ROUND(AVG(total_enter + total_exit), 1) AS avg_daily_volume
FROM vstone_catalog.gold.agg_traffic_by_location_daily
GROUP BY location, latitude, longitude
ORDER BY avg_daily_volume DESC
""")
display(traffic_by_node)

## Query 4: Traffic volume trend over time (all nodes)

In [0]:
traffic_trend = spark.sql("""
SELECT
    reading_date,
    SUM(total_enter) AS total_enter,
    SUM(total_exit) AS total_exit
FROM vstone_catalog.gold.agg_traffic_by_location_daily
GROUP BY reading_date
ORDER BY reading_date
""")
display(traffic_trend)

## Query 5: Street activity status (from the Day 7 churn metric)

In [0]:
street_status = spark.sql("""
SELECT
    churn_status,
    COUNT(*) AS street_count,
    ROUND(AVG(total_reports), 1) AS avg_reports
FROM vstone_catalog.gold.agg_stale_streets
GROUP BY churn_status
ORDER BY street_count DESC
""")
display(street_status)

## Query 6: Citizen report volume by street

In [0]:
reports_by_street = spark.sql("""
SELECT month_year, street_name, report_count
FROM vstone_catalog.gold.agg_citizen_reports_by_street
ORDER BY month_year, report_count DESC
""")
display(reports_by_street)

print("6 dashboard queries executed. See docs/day10_traffic_insights_dashboard_setup.md")
print("for building these into a Lakeview dashboard via the UI.")